## Imports

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

---
## Data Example

In [2]:
df = pd.read_csv('../../data/Advertising.csv')

In [3]:
df.head()

,TV,radio,newspaper,sales
0,230.1,37.8,69.2,22.1
1,44.5,39.3,45.1,10.4
2,17.2,45.9,69.3,9.3
3,151.5,41.3,58.5,18.5
4,180.8,10.8,58.4,12.9


---
## Train | Test Split Procedure 

* Clean and adjust data as necessary for X and y
* Split Data in Train/Test for both X and y
* Fit/Train Scaler on Training X Data
* Scale X Test Data
* Create Model
* Fit/Train Model on X Train Data
* Evaluate Model on X Test Data (by creating predictions and comparing to Y_test)
* Adjust Parameters as Necessary and repeat steps 5 and 6

In [40]:
## CREATE X and y
X = df.drop('sales',axis=1)
y = df['sales']

# TRAIN TEST SPLIT
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=101)

# SCALE DATA
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
scaler.fit(X_train)
X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)

---
**Create Model**

In [29]:
from sklearn.linear_model import Ridge

In [30]:
model = Ridge(alpha=100)

In [32]:
model.fit(X_train, y_train)

,alpha,100
,fit_intercept,True
,copy_X,True
,max_iter,None
,tol,0.0001
,solver,'auto'
,positive,False
,random_state,None


In [33]:
y_pred = model.predict(X_test)

---
**Evaluation**

In [34]:
from sklearn.metrics import mean_squared_error

In [35]:
mean_squared_error(y_test, y_pred)

7.34177578903413

---
## Train | Validation | Test Split Procedure 

This is often also called a "hold-out" set, since we should not adjust parameters based on the final test set, but instead use it *only* for reporting final expected performance.

* Clean and adjust data as necessary for X and y
* Split Data in Train/Validation/Test for both X and y
* Fit/Train Scaler on Training X Data
* Scale X Eval Data
* Create Model
* Fit/Train Model on X Train Data
* Evaluate Model on X Evaluation Data (by creating predictions and comparing to Y_eval)
* Adjust Parameters as Necessary and repeat steps 5 and 6
* Get final metrics on Test set (not allowed to go back and adjust after this!)

In [41]:
## CREATE X and y
X = df.drop('sales',axis=1)
y = df['sales']

In [42]:
######################################################################
#### SPLIT TWICE! Here we create TRAIN | VALIDATION | TEST  #########
####################################################################
from sklearn.model_selection import train_test_split

# 70% of data is training data, set aside other 30%
X_train, X_other, y_train, y_other = train_test_split(X, y, test_size=0.3, random_state=101)

# Remaining 30% is split into evaluation and test sets
# Each is 15% of the original data size
X_eval, X_test, y_eval, y_test = train_test_split(X_other, y_other, test_size=0.5, random_state=101)

In [44]:
# SCALE DATA
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
scaler.fit(X_train)
X_train = scaler.transform(X_train)
X_eval = scaler.transform(X_eval)
X_test = scaler.transform(X_test)

---
**Create Model**

In [45]:
from  sklearn.linear_model import Ridge

In [46]:
# Poor Alpha Choice on purpose!
model_one = Ridge(alpha=100)

In [47]:
model.fit(X_train, y_train)

,alpha,100
,fit_intercept,True
,copy_X,True
,max_iter,None
,tol,0.0001
,solver,'auto'
,positive,False
,random_state,None


In [48]:
y_eval_pred = model.predict(X_eval)

---
**Evaluation**

In [49]:
from sklearn.metrics import mean_squared_error

In [50]:
mean_squared_error(y_eval, y_eval_pred)

7.320101458823871

---
**Adjust Parameters and Re-evaluate**

In [51]:
model = Ridge(alpha=1)

In [53]:
model.fit(X_train, y_train)

,alpha,1
,fit_intercept,True
,copy_X,True
,max_iter,None
,tol,0.0001
,solver,'auto'
,positive,False
,random_state,None


In [54]:
y_eval_pred = model.predict(X_eval)

---
**Another Evaluation**

In [55]:
mean_squared_error(y_eval, y_eval_pred)

2.383783075056986

---
**Final Evaluation (Can no longer edit parameters after this!)**

In [56]:
y_final_test_pred = model.predict(X_test)

In [57]:
mean_squared_error(y_test, y_final_test_pred)

2.2542600838005176

---
## Cross Validation with cross_val_score

---

<img src="grid_search_cross_validation.png">

---

In [58]:
## CREATE X and y
X = df.drop('sales',axis=1)
y = df['sales']

# TRAIN TEST SPLIT
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=101)

# SCALE DATA
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
scaler.fit(X_train)
X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)

In [59]:
model = Ridge(alpha=100)

In [60]:
from sklearn.model_selection import cross_val_score

In [61]:
scores = cross_val_score(model, X_train, y_train, scoring='neg_mean_squared_error', cv=5)

In [62]:
scores

array([ -9.32552967,  -4.9449624 , -11.39665242,  -7.0242106 ,
        -8.38562723])

In [63]:
# Average of the MSE scores (we set back to positive)
abs(scores.mean())

np.float64(8.215396464543607)

---
**Adjust model based on metrics**

In [64]:
model = Ridge(alpha=1)

In [65]:
scores = cross_val_score(model,X_train,y_train,scoring='neg_mean_squared_error',cv=5)

In [66]:
# Average of the MSE scores
abs(scores.mean())

np.float64(3.344839296530695)

---
**Final Evaluation (Can no longer edit parameters after this!)**

In [67]:
# Need to fit the model first!
model.fit(X_train,y_train)

,alpha,1
,fit_intercept,True
,copy_X,True
,max_iter,None
,tol,0.0001
,solver,'auto'
,positive,False
,random_state,None


In [68]:
y_final_test_pred = model.predict(X_test)

In [69]:
mean_squared_error(y_test,y_final_test_pred)

2.319021579428752

---
# Cross Validation with cross_validate

The cross_validate function differs from cross_val_score in two ways:

It allows specifying multiple metrics for evaluation.

It returns a dict containing fit-times, score-times (and optionally training scores as well as fitted estimators) in addition to the test score.

For single metric evaluation, where the scoring parameter is a string, callable or None, the keys will be:
        
        - ['test_score', 'fit_time', 'score_time']

And for multiple metric evaluation, the return value is a dict with the following keys:

    ['test_<scorer1_name>', 'test_<scorer2_name>', 'test_<scorer...>', 'fit_time', 'score_time']

return_train_score is set to False by default to save computation time. To evaluate the scores on the training set as well you need to be set to True.

In [70]:
## CREATE X and y
X = df.drop('sales',axis=1)
y = df['sales']

# TRAIN TEST SPLIT
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=101)

# SCALE DATA
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
scaler.fit(X_train)
X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)

In [71]:
model = Ridge(alpha=100)

In [72]:
from sklearn.model_selection import cross_validate

In [73]:
scores = cross_validate(model, X_train, y_train, scoring=['neg_mean_absolute_error', 'neg_mean_squared_error', 'max_error'], cv=5)

In [74]:
scores

{'fit_time': array([0.00085759, 0.00057077, 0.00048351, 0.00059772, 0.00055075]),
 'score_time': array([0.00063658, 0.00055909, 0.00055718, 0.00055575, 0.00053859]),
 'test_neg_mean_absolute_error': array([-2.31243044, -1.74653361, -2.56211701, -2.01873159, -2.27951906]),
 'test_neg_mean_squared_error': array([ -9.32552967,  -4.9449624 , -11.39665242,  -7.0242106 ,
         -8.38562723]),
 'test_max_error': array([ -6.44988486,  -5.58926073, -10.33914027,  -6.61950405,
         -7.75578515])}

In [75]:
pd.DataFrame(scores)

,fit_time,score_time,test_neg_mean_absolute_error,test_neg_mean_squared_error,test_max_error
0,0.000858,0.000637,-2.312430,-9.325530,-6.449885
1,0.000571,0.000559,-1.746534,-4.944962,-5.589261
2,0.000484,0.000557,-2.562117,-11.396652,-10.339140
3,0.000598,0.000556,-2.018732,-7.024211,-6.619504
4,0.000551,0.000539,-2.279519,-8.385627,-7.755785


In [76]:
pd.DataFrame(scores).mean()

fit_time                        0.000612
score_time                      0.000569
test_neg_mean_absolute_error   -2.183866
test_neg_mean_squared_error    -8.215396
test_max_error                 -7.350715
dtype: float64

---
**Adjust model based on metrics**

In [77]:
model = Ridge(alpha=1)

In [79]:
scores = cross_validate(model, X_train, y_train, scoring=['neg_mean_absolute_error', 'neg_mean_squared_error', 'max_error'], cv=5)

In [80]:
pd.DataFrame(scores).mean()

fit_time                        0.000624
score_time                      0.000662
test_neg_mean_absolute_error   -1.319685
test_neg_mean_squared_error    -3.344839
test_max_error                 -5.161145
dtype: float64

---
**Final Evaluation (Can no longer edit parameters after this!)**

In [81]:
model.fit(X_train, y_train)

,alpha,1
,fit_intercept,True
,copy_X,True
,max_iter,None
,tol,0.0001
,solver,'auto'
,positive,False
,random_state,None


In [82]:
y_final_test_pred = model.predict(X_test)

In [83]:
mean_squared_error(y_test, y_final_test_pred)

2.319021579428752